# From linear classification to neural networks

**Lab 5 · Notebook 1 of 3**

Work through this notebook, then **`Week_05_nns_with_torchnn.ipynb`**, then **`Week_05_nn_exercises.ipynb`**. Use **Google Colab** and run the cells from top to bottom. Each notebook creates the same data, and no GPU is needed.

We first apply our linear classifier from Lab 4 to a new dataset. After examining its loss, mistakes and decision boundaries, we ask whether a neural network can fit these observations better. We will then compare the same quantities for the two models.

At each question, write down your answer and its reason before opening **Show answer** or **Show explanation**. For the coding exercise, try the task in its code cell first. Opening an answer does not execute its code: copy the supplied code into the exercise cell if you need it, then continue.

## 1. Generate and visualise the data

We generate **100 observations**, arranged in four Gaussian clouds. Each cloud has 25 observations, with independent coordinate noise of standard deviation $0.45$.

| Cloud centre | Class | Observations |
|---|---:|---:|
| $(-1,-1)$ | 0 | 25 |
| $(1,1)$ | 0 | 25 |
| $(-1,1)$ | 1 | 25 |
| $(1,-1)$ | 1 | 25 |

Diagonally opposite clouds share a label. This is a noisy version of the XOR arrangement from the lectures: the inputs are real-valued, and each label records the class assigned to the cloud that generated the observation. Blue circles show class 0 and orange triangles show class 1, as in Lab 4.

The tensor `X` contains the two input coordinates and has shape `(N, 2)`. The labels in `y` have shape `(N, 1)`. Run the supplied cell to create and visualise the data.


In [ ]:
import torch as t
import matplotlib.pyplot as plt
t.manual_seed(1)  # Fix the random seed for this notebook.

N = 100
# Four Gaussian clouds in an XOR arrangement: 25 observations per cloud.
# spread is the standard deviation of each coordinate, not a variance.
spread = 0.45
x_class_0 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0, -1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0,  1.0])
], 0)
x_class_1 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0,  1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0, -1.0])
], 0)
X = t.cat([x_class_0, x_class_1], 0)              # Shape: (100, 2).
y = t.cat([t.zeros(N // 2, 1), t.ones(N // 2, 1)], 0)  # Shape: (100, 1).

# Show observed classes using the same colours and markers as in Lab 4.
plt.figure()
plt.scatter(x=x_class_0[:, 0], y=x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x=x_class_1[:, 0], y=x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.axis('equal')
plt.legend()
plt.show()


## 2. Reuse the linear classifier from Lab 4

Look at the four clouds. **Could a single straight decision boundary separate the four centres according to their labels?** Keep your prediction in mind while running the model.

The code below supplies the completed calculations from Lab 4: the stable empirical logistic loss, its gradient and the number of classification mistakes. You do not need to implement them again. Recall why we use `logsigmoid` to evaluate the logarithms directly.

As in Lab 4, we include the intercept using a column of ones. Here that design matrix is called `X_affine`, while `X` keeps the original coordinates for the later network. The functions are prefixed with `affine_` to keep the two calculations separate. Their mathematical definitions are unchanged.

The coefficient column contains $(b,w_1,w_2)$, and the scores are `X_affine @ theta`. The predicted label is 1 when the sigmoid of the score is greater than 0.5, and 0 otherwise, including at equality. This is equivalent to testing whether the score is strictly positive.


In [ ]:
# Supplied: the completed functions from Lab 4, with affine-specific names.
X_affine = t.cat([t.ones(N, 1), X], dim=1)  # Shape (N, 3), including the intercept.

def affine_loss(theta):
    scores = X_affine @ theta
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

def affine_grad(theta):
    scores = X_affine @ theta
    probabilities = t.sigmoid(scores)
    return X_affine.mT @ (probabilities - y)

def affine_mistakes(theta):
    probabilities = t.sigmoid(X_affine @ theta)
    predicted_labels = (probabilities > 0.5).to(y.dtype)
    return int((predicted_labels != y).sum().item())


### 2.1. Run gradient descent

Run the supplied update loop. We use Lab 4's zero initialisation and learning rate `0.01`, now allowing **1,000 updates**. Every update uses the full dataset.

As before, record the initial loss and mistake count, then record them again **after** each update. Entry `step` in each history therefore describes exactly `step` completed updates. We also keep copies of the parameters at selected steps for the boundary plot.


In [ ]:
affine_theta = t.zeros(3, 1)
affine_learning_rate = 0.01
num_updates = 1000
affine_loss_history = [affine_loss(affine_theta).item()]
affine_error_history = [affine_mistakes(affine_theta)]
affine_checkpoints = {}

for step in range(1, num_updates + 1):
    affine_theta = affine_theta - affine_learning_rate * affine_grad(affine_theta)
    affine_loss_history.append(affine_loss(affine_theta).item())
    affine_error_history.append(affine_mistakes(affine_theta))

    # Supplied: save independent copies for the later boundary plot.
    if step in (1, 10, 100, num_updates):
        affine_checkpoints[step] = affine_theta.clone()

print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, num_updates):
    print(f'{step:7d}   {affine_loss_history[step]:23.6f}   {affine_error_history[step]:8d}')


### 2.2. Inspect the curves and the boundaries

Run the supplied plotting cells. These are the same quantities we examined in Lab 4:

- the summed empirical logistic loss;
- the loss divided by $\log 2$, alongside the number of mistakes;
- the decision boundaries at successive updates.

The division by $\log 2$ is only for display: **the updates use the original, unscaled loss**. Each mistake contributes at least $\log 2$ to that loss, so the scaled loss bounds the mistake count from above.

All affine boundaries have the form $b+w_1x_1+w_2x_2=0$. For these saved coefficients, $w_2$ is nonzero, so we can draw the lines using $x_2=-(b+w_1x_1)/w_2$, just as in Lab 4. This plotting code is supplied; focus on interpreting its output.


In [ ]:
import math

updates = range(num_updates + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(updates, affine_loss_history, color='0.35')
axes[0].set_title('Affine model: empirical logistic loss')
axes[0].set_ylabel('Summed loss')
axes[0].set_ylim(0, 75)  # Keep the small change in perspective; the table gives exact values.

axes[1].plot(updates, [value / math.log(2) for value in affine_loss_history],
             color='tab:purple', label=r'Loss / $\log 2$')
axes[1].plot(updates, affine_error_history, color='tab:green', label='Mistakes')
axes[1].set_title('Affine model: loss and mistakes')
axes[1].set_ylabel('Value')
axes[1].set_ylim(0, 110)
axes[1].legend()
for ax in axes:
    ax.set_xlabel('Completed updates')
    ax.grid(alpha=0.2)
plt.show()


In [ ]:
# The line-plotting calculation from Lab 4, using the new observations.
x1_limits = (X[:, 0].min().item() - 0.5, X[:, 0].max().item() + 0.5)
x2_limits = (X[:, 1].min().item() - 0.5, X[:, 1].max().item() + 0.5)
x1_line = t.tensor(x1_limits)

fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
plt.scatter(x_class_0[:, 0], x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x_class_1[:, 0], x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
for step, style in zip((1, 10, 100, num_updates), (':', '--', '-.', '-')):
    b, w1, w2 = affine_checkpoints[step][:, 0]
    x2_line = -(b + w1 * x1_line) / w2
    plt.plot(x1_line, x2_line, color='black', linestyle=style,
             label=f'After {step} update' + ('s' if step != 1 else ''))
plt.xlim(x1_limits)
plt.ylim(x2_limits)
plt.gca().set_aspect('equal')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Affine decision boundaries')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()


### Pause and interpret: what has gradient descent achieved?

1. Compare the initial and final empirical losses and mistake counts. Has the affine model fitted the observations well? The loss curve looks almost flat: does this mean that no updates have taken place? Use the printed values to read changes that are too small to see in the curves.
2. What happens to the boundary between updates 1, 100 and 1,000? Could more updates turn it into a boundary with several segments?
3. Why do the loss and the number of mistakes not necessarily decrease together?

Write down your answers before opening the explanation.


<details>
<summary>Show answer — interpreting the affine fit</summary>

**1. The loss decreases very little, and many mistakes remain.** With the supplied seed, the loss changes from approximately $69.3147$ to $69.3083$, while the mistake count changes from 50 to 47. The loss has already essentially stopped changing by update 100. The fitted model still misclassifies almost half of these observations.

The curve looks flat because the total decrease is only about $0.00645$. Initially, all coefficients are zero, so every score is zero and every predicted probability is $1/2$. The summed empirical loss is therefore $N\log 2\approx69.3147$.

For this nearly symmetric XOR sample, the fitted affine scores remain close to zero: a single straight decision boundary cannot capture the main XOR pattern. Small sample asymmetries allow a slight improvement, including a few changed class predictions, but the probabilities remain near $1/2$. The empirical logistic loss is convex in the affine coefficients, so this behaviour is not due to a suboptimal local minimum. It illustrates a limitation of the affine model on these data.

**2. Updates change the coefficients of a straight line.** The boundary initially rotates and shifts; the lines at 100 and 1,000 updates almost coincide. Further updates can change its slope and intercept, but an affine score always has a straight boundary when its feature weights are nonzero. A single straight line cannot separate the four cloud centres according to their XOR labels. We need a richer family of scores to express a different shape.

**3. The two quantities measure different things.** The logistic loss depends on the probabilities assigned to the observed labels. The mistake count only checks which side of the classification threshold each observation falls on. Probabilities can change without changing any labels, and an improvement in total loss can occur even when the number of mistakes increases.

The 47 mistakes are the outcome of fitting the logistic loss in this run. They are not a claim that 47 is the smallest mistake count attainable by every possible straight boundary.

</details>


## 3. Can we do better with a neural network?

We now keep the **same observations, labels and empirical logistic loss**, and change the model used to produce the scores. In the lectures, we introduced neural networks so that the feature functions themselves could be adjusted through hidden weights and biases. We will now fit those parameters and see whether this improves the classification of our four clouds.

We will calculate the network outputs and update its parameters explicitly. In Lab 4, we wrote the gradient formula ourselves. Here, PyTorch computes the gradients for us using **automatic differentiation**. Calling `L.backward()` computes the derivatives of the scalar loss `L` with respect to the network parameters and stores them in their `.grad` attributes. We then use these derivatives in the same gradient-descent update as before. In the next lecture, we will study **backpropagation**, the algorithm behind this calculation.

### The same empirical logistic loss

In Lab 4, our loss function computed the affine scores internally. Here, we calculate the network scores separately and pass them to `loss_from_scores`, which evaluates the same empirical logistic loss. This lets us reuse the loss calculation with the different network implementations introduced later. We could also use this function with affine scores.

The function uses the observed labels stored in `y`, which we defined with the dataset.

Recall the numerical stability problem encountered in Lab 4. For the same reason, we use `logsigmoid` to evaluate the required logarithms directly. Pass the network scores to `loss_from_scores` without applying a sigmoid first.


In [ ]:
def loss_from_scores(scores):
    # scores contains classification scores, not probabilities or weights.
    # Evaluate sum_i ell_log(sigmoid(score_i), y_i), using the fixed labels y.
    # logsigmoid computes the logarithms directly, without forming probabilities.
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()


### 3.1. Define the network and its parameters

We use a network with one hidden layer and one output layer. Each layer keeps its weights and biases separate, as in the notes. Therefore, `X` contains only the original coordinates; the added column of ones belongs only to `X_affine`. The hidden layer applies an affine transformation followed by ReLU. The output layer then combines the hidden activations through a second affine transformation. We number these layers from 1, so their parameters are called `W1`, `b1`, `W2`, and `b2`.

As in the lecture notes, we write $\operatorname{ReLU}(a)=\max\{0,a\}$ for the scalar function and use bold type, $\operatorname{\mathbf{ReLU}}$, for the vector transformation that applies it separately to each coordinate. For a single input row $\boldsymbol{x}$, the network computes

$$
\begin{aligned}
\boldsymbol{a}_1(\boldsymbol{x}) &= \boldsymbol{x}\boldsymbol{W}_1+\boldsymbol{b}_1,\\
\boldsymbol{\phi}_1(\boldsymbol{x}) &= \operatorname{\mathbf{ReLU}}\bigl(\boldsymbol{a}_1(\boldsymbol{x})\bigr),\\
\boldsymbol{f}_{\boldsymbol{\theta}}(\boldsymbol{x}) &= \boldsymbol{\phi}_1(\boldsymbol{x})\boldsymbol{W}_2+\boldsymbol{b}_2.
\end{aligned}
$$

The map $\boldsymbol{\phi}_1$ plays the same role as the feature map $\boldsymbol{\phi}$ in Lecture 2, although its feature functions now depend on the hidden layer's adjustable weights and biases. As in the lecture notes, we use $\phi$ for these features and reserve $h$ for classifiers. The symbol $\boldsymbol{\theta}$ collects all the parameters mathematically. In the code, we work directly with the four tensors `W1`, `b1`, `W2`, and `b2`.

The notes describe one input row at a time. In this notebook, `X` contains all `N` observations, one per row, and the same parameters are used for every observation. Consequently, `a1` and `phi1` will also contain one row per observation. These variables store the values computed for the current inputs. The output variable `scores` contains one classification score per observation.

In the code, `t` is our alias for `torch`, and `t.relu(a1)` applies ReLU to every entry of the tensor `a1`. For a single row, this implements the vector transformation $\operatorname{\mathbf{ReLU}}$. For a batch of rows, it performs the same calculation on each row. The distinction between normal and bold type belongs to our mathematical notation, while PyTorch uses the same function name for tensors of different shapes.

Our network has two input features, 100 hidden units, and one output unit. Its tensors therefore have the following shapes:

| Tensor | Meaning | Shape |
|---|---|---|
| `X` | Input observations | `(N, 2)` |
| `W1` | Hidden-layer weights | `(2, 100)` |
| `b1` | Hidden-layer biases | `(100,)` |
| `a1`, `phi1` | Hidden preactivations and activations | `(N, 100)` |
| `W2` | Output-layer weights | `(100, 1)` |
| `b2` | Output-layer bias | `(1,)` |
| `scores` | Network output | `(N, 1)` |

In the notes, biases are written as rows. Here we store each bias vector in a one-dimensional tensor. PyTorch adds that same vector to every row of the matrix through **broadcasting**, which gives the same affine calculation for each observation.

The last layer returns **scores, also called logits**. We pass them to `loss_from_scores`, which evaluates the sum of $\ell_{\mathrm{log}}(\sigma(s_i),y_i)$ directly from those scores using `logsigmoid`. We therefore do not add a separate sigmoid before this call.

When we create a parameter tensor, **`requires_grad=True`** tells PyTorch to track computations involving it so that we can later calculate the loss gradient with respect to that tensor. It does not calculate the gradient or update the parameter yet.


In [ ]:
input_features = 2
hidden_features = 100
output_features = 1

# Parameters of hidden layer 1: 2 input features -> 100 hidden activations.
b1 = t.randn(hidden_features, requires_grad=True)                     # (100,)
W1 = t.randn(input_features, hidden_features, requires_grad=True)     # (2, 100)

# Parameters of output layer 2: 100 hidden activations -> 1 score.
b2 = t.randn(output_features, requires_grad=True)                     # (1,)
W2 = t.randn(hidden_features, output_features, requires_grad=True)    # (100, 1)

def net(X):
    """Apply the same two-layer network to every input row."""
    # The parameters above are reused on every call, not randomly redrawn.
    a1 = X @ W1 + b1       # Hidden preactivations: (N, 100) for the dataset X.
    phi1 = t.relu(a1)        # Hidden activations: (N, 100).
    scores = phi1 @ W2 + b2  # Output scores: (N, 1).
    return scores


### 3.2. Compute the scores and the loss

`net(X)` computes one score for each observation. The call `loss_from_scores(scores)` then evaluates $\ell_{\mathrm{log}}(\sigma(s_i),y_i)$ for each observation and adds these values using `.sum()`. The result is therefore a **scalar tensor**. The labels come from the existing tensor `y` and remain fixed.

When you print `L`, the extra `grad_fn=...` text indicates that PyTorch has recorded how it was computed. You do not need to learn the internal name shown there.
The value stored in `L` is the empirical objective evaluated at the current parameters:

$$
\mathcal{L}_{\mathrm{log}}(\boldsymbol{\theta})=\sum_{i=1}^N\ell_{\mathrm{log}}\bigl(\sigma(f_{\boldsymbol{\theta}}(\boldsymbol{x}_i)),y_i\bigr).
$$

The objective takes the parameters as its argument. The helper takes the scores computed from those parameters, with the observed labels fixed.


In [ ]:
scores = net(X)
print(scores.shape)  # torch.Size([100, 1]), matching the shape of y.
L = loss_from_scores(scores)
print(L)

### 3.3. Compute the gradients with `backward()`

Calling **`L.backward()`** computes the derivatives of this scalar loss with respect to our four parameter tensors. PyTorch stores them in each parameter's **`.grad` attribute**.

For example, `b1.grad[0]` contains the derivative of the total loss with respect to the first hidden-layer bias, written $(\boldsymbol{b}_1)_1$ in the notes. The `1` in the name `b1` identifies the first layer, while `[0]` selects the first entry using Python's indexing convention. The gradient tensor has the same shape as the parameter tensor. Before the first call to `L.backward()`, these `.grad` attributes are `None` because no gradient has been stored yet.

**`backward()` does not update the parameters.** It also **adds to any gradients already stored**, rather than replacing them. We will clear them when training below.
When computing gradients, PyTorch uses 0 in place of the undefined derivative of ReLU at input 0. This is a computational convention.


### Pause and predict: one loss, many derivatives

We have computed the network's scores and the scalar loss. Before running the next code cell, consider what `L.backward()` will change.

Will the values stored in `b1` change? What shape should `b1.grad` have after the call? Explain why it should have that shape even though the loss is a single number.


<details>
<summary>Show explanation — what changes after backward()?</summary>

The values in `b1` remain unchanged because `backward()` computes derivatives. We will use those derivatives to update the parameters in the next section.

The tensor `b1.grad` has shape `(100,)`, matching `b1`. There is one derivative for each of the 100 biases. Although the loss is a single number, it depends on all those adjustable coefficients. Each derivative describes how that same loss changes when we vary one bias while keeping the other parameters fixed.

Before the first call to `L.backward()`, `b1.grad` is `None`. After the call, it contains these derivatives, evaluated at the current parameter values. The next code cell prints them.

</details>


In [ ]:
L = loss_from_scores(net(X))  # Compute a fresh loss before calculating its gradients.
L.backward()
print(b1.grad)  # One derivative for each of the 100 hidden-layer biases.

### Check the gradients yourself

The preceding cell has now computed the gradients. Use the values already stored in `.grad` to check the other parameter tensors as well.

For each of `W1`, `b1`, `W2`, and `b2`, print the shape of the parameter and the shape of its gradient. Check whether they match, and explain why there should be one gradient entry for each parameter entry.

Now suppose that one entry of `W1.grad` is positive and that the learning rate is positive. Will the gradient-descent update increase or decrease the corresponding weight? Explain your answer using the update rule.

This check only reads the existing parameters and gradients. You can then continue directly to the training section.


In [ ]:
# Print each parameter's shape and the shape of its stored gradient.
# Explain the update direction for a positive gradient entry in a comment.


<details>
<summary>Show answer — gradient shapes and update direction</summary>

```python
print("W1:", W1.shape, "gradient:", W1.grad.shape)
print("b1:", b1.shape, "gradient:", b1.grad.shape)
print("W2:", W2.shape, "gradient:", W2.grad.shape)
print("b2:", b2.shape, "gradient:", b2.grad.shape)
```

The output should be:

```text
W1: torch.Size([2, 100]) gradient: torch.Size([2, 100])
b1: torch.Size([100]) gradient: torch.Size([100])
W2: torch.Size([100, 1]) gradient: torch.Size([100, 1])
b2: torch.Size([1]) gradient: torch.Size([1])
```

For every parameter tensor, each gradient entry gives the derivative of the scalar loss with respect to the parameter entry in the same position. Therefore, the shapes agree. For example, `W1.grad[0, 0]` corresponds to `W1[0, 0]`.

With a positive learning rate, the update subtracts `learning_rate * gradient_entry` from the corresponding weight. If that gradient entry is positive, we subtract a positive number, so the weight decreases. This is the component-by-component version of the gradient-descent rule from the lectures.

We use the gradients computed above with `L.backward()`. Printing their shapes does not change them. Before computing new gradients, the fitting loop below clears these stored values so that successive calculations do not accumulate.

</details>


### 3.4. Use those gradients for gradient descent

We keep the same update rule: subtract `learning_rate * gradient` from each parameter. Each iteration computes a fresh loss, calls `backward()`, updates the parameters, and clears the gradients for the next iteration.

**`with t.no_grad():`** turns off gradient recording only for its indented block. Here we have already computed the gradients: we now want to change the parameters without recording the update itself as part of the next loss calculation. Recording resumes after the block.

**`.add_(...)` changes a tensor in place**. In contrast, **`b1.grad.zero_()` clears the stored gradient while leaving the parameter values in `b1` unchanged**.

The earlier call to `L.backward()` already computed gradients for demonstration. Clear those once before starting the loop, so the first update uses only the gradients computed in that iteration.

We will run **1,000 updates**, using the full dataset at every step. We also record the empirical logistic loss and the number of mistakes before the first update and after each update, just as in Lab 4. The network uses learning rate `0.0001`; the affine comparison uses `0.01`. These are separate choices for two different parameterisations, not a comparison of optimisation speed.

The following setup creates a grid and a dictionary for storing boundary snapshots. **This plotting setup is non-examinable.** It does not change the data or the parameters. Focus on the gradient update and on what the recorded loss and mistake count tell us.


In [ ]:
# Non-examinable plotting setup: evaluate scores on the same grid at each checkpoint.
x1_limits = (X[:, 0].min().item() - 0.5, X[:, 0].max().item() + 0.5)
x2_limits = (X[:, 1].min().item() - 0.5, X[:, 1].max().item() + 0.5)
x1_grid, x2_grid = t.meshgrid(t.linspace(*x1_limits, 250),
                            t.linspace(*x2_limits, 250), indexing="xy")
X_grid = t.stack([x1_grid, x2_grid], dim=-1).reshape(-1, 2)
boundary_steps = (1, 100, num_updates)
boundary_scores = {}


### Pause and reason: what remains between iterations?

The next code cell clears the gradients before training and again at the end of each iteration. Read the loop before running it, and consider this hypothetical change: we keep the initial reset but remove the four `.grad.zero_()` calls at the end of the loop.

After the first iteration, what would remain stored in `.grad`? When the second call to `backward()` adds the newly computed gradient, what would `.grad` contain? Would the second update use only the gradient of the loss at the current parameters?

Write down your reasoning, then use the explanation below to check it. Run the supplied training loop with its resets in place.


<details>
<summary>Show explanation — why the loop clears gradients</summary>

The initial reset leaves zero in each stored gradient tensor. Therefore, the first call to `backward()` stores the gradient at the initial parameters, and the first update uses that gradient.

Updating the parameters does not clear their `.grad` tensors. If we omit the resets at the end of the first iteration, those tensors still contain the gradient used for the first update.

The second iteration computes a new loss at the updated parameters. Its call to `backward()` adds the new gradient to the stored one. Consequently, `.grad` now contains the sum of gradients evaluated at two different sets of parameter values, and the second update uses that sum.

Our gradient-descent loop is intended to use the gradient at the current parameters in each iteration. Clearing `.grad` after each update ensures that the next call to `backward()` stores just the newly computed gradient.

</details>


In [ ]:
# Remove the gradients left by the preceding demonstration.
b1.grad.zero_()
W1.grad.zero_()
b2.grad.zero_()
W2.grad.zero_()

with t.no_grad():
    scores = net(X)
    loss_history = [loss_from_scores(scores).item()]
    error_history = [int(((scores > 0) != y).sum().item())]

learning_rate = 0.0001
for step in range(1, num_updates + 1):
    L = loss_from_scores(net(X))
    L.backward()

    # Update the parameters without recording the update for differentiation.
    with t.no_grad():
        b1.add_(-learning_rate * b1.grad)
        W1.add_(-learning_rate * W1.grad)
        b2.add_(-learning_rate * b2.grad)
        W2.add_(-learning_rate * W2.grad)

        # Measure AFTER this update. Entry 0 records the initial parameters.
        scores = net(X)
        loss_history.append(loss_from_scores(scores).item())
        error_history.append(int(((scores > 0) != y).sum().item()))

        # Non-examinable: save the score values now, before the parameters change again.
        if step in boundary_steps:
            boundary_scores[step] = net(X_grid).reshape(x1_grid.shape).clone()

    # Clear the stored gradients before the next gradient calculation.
    b1.grad.zero_()
    W1.grad.zero_()
    b2.grad.zero_()
    W2.grad.zero_()

    if step % 100 == 0:
        print(f"After {step} updates: loss = {loss_history[-1]:.3f}, "
              f"mistakes = {error_history[-1]}/{N}")


## 4. Inspect the network and compare the two models

### 4.1. Plot the loss and the number of mistakes

Run the supplied plotting cell. The first panel compares the empirical logistic losses. The other panels show the loss divided by $\log 2$ and the mistake count for each model, with the same axes. We have used the same data and loss throughout; the two models have different parameterisations and learning rates, so these curves are not a controlled comparison of optimisation speed.

The histories record the initial state at entry 0 and the state after each update. You can inspect particular entries when a change is difficult to see in the plot. The plotting code is supplied and non-examinable.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
axes[0].plot(updates, affine_loss_history, '--', color='0.4', label='Affine score')
axes[0].plot(updates, loss_history, color='tab:purple', label='Neural network')
axes[0].set_title('Empirical logistic loss')
axes[0].set_ylabel('Summed loss')
axes[0].legend()

for ax, losses, errors, title in (
    (axes[1], affine_loss_history, affine_error_history, 'Affine model'),
    (axes[2], loss_history, error_history, 'Neural network')
):
    ax.plot(updates, [value / math.log(2) for value in losses],
            color='tab:purple', label=r'Loss / $\log 2$')
    ax.plot(updates, errors, color='tab:green', label='Mistakes')
    ax.set_title(title + ': loss and mistakes')
    ax.set_ylabel('Value')
    ax.set_ylim(0, 235)
    ax.legend()
for ax in axes:
    ax.set_xlabel('Completed updates')
    ax.grid(alpha=0.2)
plt.show()

print('After 1,000 updates, on the same 100 observations:')
print(f'Affine model: loss = {affine_loss_history[-1]:.3f}, mistakes = {affine_error_history[-1]}')
print(f'Neural network: loss = {loss_history[-1]:.3f}, mistakes = {error_history[-1]}')


### 4.2. See how the decision boundary changes

The black line shows where the score equals zero. Light blue regions receive label 0; light orange regions receive label 1. The markers show the observed classes. A score of exactly zero receives label 0, as in Lab 4.

Compare the fitted affine boundary with snapshots of the network after 1, 100 and 1,000 updates. The axes and the observations are the same in all four plots. **The plotting code is non-examinable**: it evaluates the score on a grid and draws the zero contour; you do not need to reproduce its implementation.

Which changes in the boundary account for the reduced mistake count? Relate this to the hidden weights and biases: we supplied the two input coordinates, while the network adjusted the feature functions used by its affine output layer.


In [ ]:
def draw_boundary(ax, score_grid, title):
    # Non-examinable helper: shade predicted classes and draw the score = 0 contour.
    classes = (score_grid > 0).to(t.float32)
    ax.contourf(x1_grid, x2_grid, classes,
                levels=[-0.5, 0.5, 1.5], colors=['#edf3fb', '#fff1e9'])
    ax.contour(x1_grid, x2_grid, score_grid, levels=[0], colors='black', linewidths=1)
    class_0 = y[:, 0] == 0
    class_1 = y[:, 0] == 1
    ax.scatter(X[class_0, 0], X[class_0, 1], color='tab:blue', s=18, label='Class 0')
    ax.scatter(X[class_1, 0], X[class_1, 1], color='tab:orange', marker='^', s=20, label='Class 1')
    ax.set(xlim=x1_limits, ylim=x2_limits, xlabel='$x_1$', ylabel='$x_2$', title=title)
    ax.set_aspect('equal')

fig, axes = plt.subplots(1, 4, figsize=(15, 4), constrained_layout=True)
affine_grid = t.cat([t.ones(X_grid.shape[0], 1), X_grid], dim=1)
affine_grid_scores = (affine_grid @ affine_theta).reshape(x1_grid.shape)
draw_boundary(axes[0], affine_grid_scores,
              f'Affine: {num_updates} updates\n{affine_error_history[-1]} mistakes')
for ax, step in zip(axes[1:], (1, 100, num_updates)):
    draw_boundary(ax, boundary_scores[step],
                  f'NN: {step} update' + ('s' if step != 1 else '')
                  + f'\n{error_history[step]} mistakes')
axes[0].legend(loc='upper left', fontsize=8)
plt.show()


### Pause and interpret: what did the network add?

1. Compare the final empirical loss and mistake count of the two models. Which changes in the boundary help the network classify the four clouds?
2. Which parameters change the feature functions, and which parameters combine their values into an output score? Did we supply the XOR feature functions ourselves?
3. Does every decrease in loss reduce the mistake count? Inspect `loss_history[81]`, `loss_history[82]`, `error_history[81]` and `error_history[82]`.
4. What do these results tell us about fitting these observations? Do they establish the best possible network fit or its accuracy on unseen observations?

Write down your answers before opening the explanation.


<details>
<summary>Show answer — comparing the affine model and the network</summary>

**1. The network fits these observations better in this run.** After 1,000 updates, the affine model has loss about $69.308$ and makes 47 mistakes. The network has loss about $8.444$ and makes 3 mistakes. Its mistake count falls from 30 initially to 7 after 100 updates and 3 after 1,000. Its boundary changes direction and creates regions that follow the four-cloud arrangement more closely.

**2. The hidden parameters determine the features.** `W1` and `b1` define the functions whose values are stored in `phi1`. The output parameters `W2` and `b2` combine those activations into scores. We supplied the two raw coordinates and the network architecture; gradient descent adjusted all four parameter tensors. We did not specify the XOR feature functions by hand.

**3. Loss reduction and mistake reduction are different.** Between updates 81 and 82, the loss decreases from about $20.514$ to $20.398$, while the number of mistakes increases from 7 to 8. The loss depends on probabilities for all observations, whereas mistakes depend only on the thresholded predictions. The scaled-loss bound still holds at both steps.

**4. This is evidence about the fitted models on the observed data.** It shows that this network and fitting procedure achieve a smaller empirical loss and fewer mistakes here. It does not show that these are globally optimal network parameters, or measure prediction accuracy on unseen observations.

The numerical values above refer to a fresh top-to-bottom run with the supplied seed. Changing the initialisation or continuing an already fitted network can change the results.

</details>


We have now compared an affine score with a neural network on the same observations, and followed how the fitted scores, loss and mistakes change. The network gives us adjustable feature functions, while the empirical loss and the gradient-descent principle remain familiar.

So far we have handled each parameter tensor explicitly. That makes the calculation visible, but becomes repetitive and easy to get wrong as networks grow. In **`Week_05_nns_with_torchnn.ipynb`**, we will organise these same calculations into reusable modules. In **`Week_05_nn_exercises.ipynb`**, you will practise building and fitting the same network with those implementations.
